In [48]:
from pycaret.classification import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

PALETTE=['lightcoral', 'lightskyblue', 'gold', 'sandybrown', 'navajowhite',
        'khaki', 'lightslategrey', 'turquoise', 'rosybrown', 'thistle', 'pink']
sns.set_palette(PALETTE)
BACKCOLOR = '#f6f5f5'

from IPython.core.display import HTML



In [49]:
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv')

# X_train = pd.read_csv('../data/X_train_fg.csv')
# X_test = pd.read_csv('../data/X_test_fg.csv')
# y_train = pd.read_csv('../data/y_train.csv')
# y_test = pd.read_csv('../data/y_test.csv')



cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)


y_train_int = y_train.astype(int)

In [50]:
from pycaret.classification import setup

# Combine X_train and y_train
train_df = X_train.copy()
train_df['target'] = y_train


# Identify categorical and numeric columns
categorical_cols = [col for col in train_df.select_dtypes(include=['object', 'category']).columns if col != 'target']
numeric_cols     = [col for col in train_df.select_dtypes(include=['int64', 'float64']).columns if col != 'target']

s = setup(
    data=train_df,
    target='target',
    session_id=7010,
    fold_strategy='stratifiedkfold', #RepeatedStratifiedKFold
    fold=5,
    fold_shuffle=True,
    verbose=False,
    normalize=True,
    normalize_method='robust',
    remove_multicollinearity=True,
    multicollinearity_threshold=0.95,
    categorical_features=categorical_cols,
    numeric_features=numeric_cols,
    # test_data=test_df,
    index=False   # <-- important: ignore the original index to prevent duplicates
)


In [51]:
best_models = compare_models(
    sort='Accuracy',  # Or 'AUC', 'F1'
    n_select=5,       # Top 5 models
    fold=5,           # Cross-validation folds
    turbo=True        # Faster comparison
)
best_models

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
catboost,CatBoost Classifier,0.8507,0.8738,0.7404,0.8528,0.7916,0.6762,0.6813,0.3280
rf,Random Forest Classifier,0.8475,0.8553,0.7486,0.8383,0.7891,0.6705,0.6747,0.0460
knn,K Neighbors Classifier,0.8458,0.8483,0.7530,0.8297,0.7894,0.6683,0.6703,0.0200
gbc,Gradient Boosting Classifier,0.8411,0.8709,0.7322,0.8342,0.7794,0.6561,0.6599,0.0340
lightgbm,Light Gradient Boosting Machine,0.8411,0.8597,0.7488,0.8220,0.7826,0.6580,0.6607,0.1280
et,Extra Trees Classifier,0.8379,0.8419,0.7277,0.8325,0.7745,0.6490,0.6545,0.0400
ada,Ada Boost Classifier,0.8362,0.8713,0.7864,0.7865,0.7863,0.6535,0.6537,0.0320
xgboost,Extreme Gradient Boosting,0.8314,0.8518,0.7488,0.8012,0.7732,0.6394,0.6412,0.1840
lr,Logistic Regression,0.8218,0.8737,0.7529,0.7783,0.7642,0.6211,0.6226,0.0200
ridge,Ridge Classifier,0.8154,0.8758,0.7488,0.7679,0.7570,0.6083,0.6096,0.0120


 RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                        criterion='gini', max_depth=None, max_features='sqrt',
                        max_leaf_nodes=None, max_samples=None,
                        min_impurity_decrease=0.0, min_samples_leaf=1,
                        min_samples_split=2, min_weight_fraction_leaf=0.0,
                        monotonic_cst=None, n_estimators=100, n_jobs=-1,
                        oob_score=False, random_state=7010, verbose=0,
                        warm_start=False),
 KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
                      metric_params=None, n_jobs=-1, n_neighbors=5, p=2,
                      weights='uniform'),
 GradientBoostingClassifier(ccp_alpha=0.0, criterion='friedman_mse', init=None,
                            learning_rate=0.1, loss='log_loss', max_depth=3,
                            max_features=None, max_leaf_nodes=None,
                            min_impur

In [52]:
best_models[0]

**Train a final best model**

In [53]:
from re import X
from pycaret.classification import finalize_model, predict_model

# Choose your top model, e.g., the first in best_models
final_model = finalize_model(best_models[0])

predictions = predict_model(final_model, data=X_test)
print(predictions.columns)
submission = pd.DataFrame({
    'PassengerId': X_test['PassengerId'],  # original test IDs
    'Survived': predictions['prediction_label']        # predicted target
})

# Save to CSV
submission.to_csv('submission/pycaret_single_best_model.csv', index=False)

Index(['PassengerId', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'relatives', 'not_alone', 'Deck', 'Title', 'Age_Class',
       'Fare_Per_Person', 'prediction_label', 'prediction_score'],
      dtype='object')


**blending**

In [54]:
from pycaret.classification import tune_model, blend_models, predict_model, finalize_model, pull

# Suppose best_models contains the top models from compare_models()
for m in best_models:
    print(type(m))  # check model types if you want

# Pick the top 4 models from best_models
top4 = best_models[:4]

# Tune the top 4 models
tuned_top4 = [tune_model(m, optimize='Accuracy') for m in top4]

# Blend the tuned top 4 models
blender_top4 = blend_models(estimator_list=tuned_top4)

# Pull evaluation metrics of the blended model
results = pull()
print(results)

# Finalize the blended model on the full training set
final_blender = finalize_model(blender_top4)

# Predict on test set
df_pred = predict_model(final_blender, data=X_test)

# Prepare submission
submission = pd.DataFrame({
    'PassengerId': X_test['PassengerId'],
    'Survived': df_pred['prediction_label']
})

submission.to_csv('submission/pycaret_blended_top4.csv', index=False)


<class 'catboost.core.CatBoostClassifier'>
<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.neighbors._classification.KNeighborsClassifier'>
<class 'sklearn.ensemble._gb.GradientBoostingClassifier'>
<class 'lightgbm.sklearn.LGBMClassifier'>


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8800,0.8927,0.8333,0.8511,0.8421,0.7453,0.7455
1,0.8960,0.9085,0.7917,0.9268,0.8539,0.7740,0.7798
2,0.8720,0.9044,0.7917,0.8636,0.8261,0.7251,0.7268
3,0.8306,0.8254,0.7021,0.8250,0.7586,0.6295,0.6343
4,0.8306,0.8244,0.7083,0.8293,0.7640,0.6332,0.6380
Mean,0.8619,0.8711,0.7654,0.8592,0.8090,0.7014,0.7049
Std,0.0266,0.0381,0.0515,0.0367,0.0399,0.0593,0.0586


Fitting 5 folds for each of 10 candidates, totalling 50 fits


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8640,0.9114,0.7500,0.8780,0.8090,0.7044,0.7097
1,0.8720,0.8980,0.7292,0.9211,0.8140,0.7184,0.7298
2,0.8320,0.9027,0.7083,0.8293,0.7640,0.6349,0.6396
3,0.8306,0.8419,0.6383,0.8824,0.7407,0.6197,0.6377
4,0.8065,0.8239,0.7083,0.7727,0.7391,0.5857,0.5871
Mean,0.8410,0.8756,0.7068,0.8567,0.7734,0.6526,0.6608
Std,0.0240,0.0356,0.0376,0.0511,0.0324,0.0507,0.0521


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


...

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8400,0.8838,0.7917,0.7917,0.7917,0.6618,0.6618
1,0.8720,0.8931,0.7708,0.8810,0.8222,0.7229,0.7268
2,0.8480,0.9103,0.7708,0.8222,0.7957,0.6749,0.6758
3,0.8387,0.8232,0.7234,0.8293,0.7727,0.6486,0.6523
4,0.8306,0.8384,0.7083,0.8293,0.7640,0.6332,0.6380
Mean,0.8459,0.8698,0.7530,0.8307,0.7893,0.6683,0.6709
Std,0.0142,0.0333,0.0316,0.0287,0.0202,0.0306,0.0305


Fitting 5 folds for each of 10 candidates, totalling 50 fits


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8800,0.9030,0.8333,0.8511,0.8421,0.7453,0.7455
1,0.8880,0.8994,0.7708,0.9250,0.8409,0.7556,0.7631
2,0.8240,0.8943,0.7292,0.7955,0.7609,0.6220,0.6235
3,0.8387,0.8176,0.7021,0.8462,0.7674,0.6456,0.6522
4,0.8387,0.8214,0.7292,0.8333,0.7778,0.6521,0.6557
Mean,0.8539,0.8671,0.7529,0.8502,0.7978,0.6841,0.6880
Std,0.0253,0.0390,0.0458,0.0422,0.0361,0.0552,0.0555


      Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC
Fold                                                          
0       0.8800  0.9030  0.8333  0.8511  0.8421  0.7453  0.7455
1       0.8880  0.8994  0.7708  0.9250  0.8409  0.7556  0.7631
2       0.8240  0.8943  0.7292  0.7955  0.7609  0.6220  0.6235
3       0.8387  0.8176  0.7021  0.8462  0.7674  0.6456  0.6522
4       0.8387  0.8214  0.7292  0.8333  0.7778  0.6521  0.6557
Mean    0.8539  0.8671  0.7529  0.8502  0.7978  0.6841  0.6880
Std     0.0253  0.0390  0.0458  0.0422  0.0361  0.0552  0.0555


**stack models**

In [55]:


from pycaret.classification import stack_models, predict_model, finalize_model, pull

# Pick top 4 models from best_models (same as before)
top4 = best_models[:4]

# Tune the top 4 models (optional but recommended)
tuned_top4 = [tune_model(m, optimize='Accuracy') for m in top4]

# Stack the tuned models using a meta-model (default is Logistic Regression)
stacked_model = stack_models(estimator_list=tuned_top4, meta_model=create_model('lr'),)  
# meta_model=None -> PyCaret will choose Logistic Regression by default

# Pull evaluation metrics of stacked model
stack_results = pull()
print(stack_results)

# Finalize the stacked model on the full training set
final_stacked = finalize_model(stacked_model)

# Predict on test set
df_pred = predict_model(final_stacked, data=X_test)

# Prepare submission
submission = pd.DataFrame({
    'PassengerId': X_test['PassengerId'],
    'Survived': df_pred['prediction_label']
})

submission.to_csv('submission/pycaret_stacked_top4.csv', index=False)


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8800,0.8927,0.8333,0.8511,0.8421,0.7453,0.7455
1,0.8960,0.9085,0.7917,0.9268,0.8539,0.7740,0.7798
2,0.8720,0.9044,0.7917,0.8636,0.8261,0.7251,0.7268
3,0.8306,0.8254,0.7021,0.8250,0.7586,0.6295,0.6343
4,0.8306,0.8244,0.7083,0.8293,0.7640,0.6332,0.6380
Mean,0.8619,0.8711,0.7654,0.8592,0.8090,0.7014,0.7049
Std,0.0266,0.0381,0.0515,0.0367,0.0399,0.0593,0.0586


Fitting 5 folds for each of 10 candidates, totalling 50 fits


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8640,0.9114,0.7500,0.8780,0.8090,0.7044,0.7097
1,0.8720,0.8980,0.7292,0.9211,0.8140,0.7184,0.7298
2,0.8320,0.9027,0.7083,0.8293,0.7640,0.6349,0.6396
3,0.8306,0.8419,0.6383,0.8824,0.7407,0.6197,0.6377
4,0.8065,0.8239,0.7083,0.7727,0.7391,0.5857,0.5871
Mean,0.8410,0.8756,0.7068,0.8567,0.7734,0.6526,0.6608
Std,0.0240,0.0356,0.0376,0.0511,0.0324,0.0507,0.0521


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8160,0.8695,0.7292,0.7778,0.7527,0.6064,0.6072
1,0.8880,0.8819,0.8333,0.8696,0.8511,0.7614,0.7618
2,0.8000,0.8199,0.7500,0.7347,0.7423,0.5789,0.5790
3,0.8226,0.8006,0.7234,0.7907,0.7556,0.6167,0.6182
4,0.8548,0.8318,0.7500,0.8571,0.8000,0.6869,0.6906
Mean,0.8363,0.8408,0.7572,0.8060,0.7803,0.6501,0.6514
Std,0.0314,0.0305,0.0396,0.0505,0.0405,0.0661,0.0664


...

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8400,0.8838,0.7917,0.7917,0.7917,0.6618,0.6618
1,0.8720,0.8931,0.7708,0.8810,0.8222,0.7229,0.7268
2,0.8480,0.9103,0.7708,0.8222,0.7957,0.6749,0.6758
3,0.8387,0.8232,0.7234,0.8293,0.7727,0.6486,0.6523
4,0.8306,0.8384,0.7083,0.8293,0.7640,0.6332,0.6380
Mean,0.8459,0.8698,0.7530,0.8307,0.7893,0.6683,0.6709
Std,0.0142,0.0333,0.0316,0.0287,0.0202,0.0306,0.0305


Fitting 5 folds for each of 10 candidates, totalling 50 fits


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8400,0.8960,0.8333,0.7692,0.8000,0.6670,0.6685
1,0.8720,0.9064,0.7917,0.8636,0.8261,0.7251,0.7268
2,0.7760,0.8643,0.7292,0.7000,0.7143,0.5302,0.5305
3,0.8145,0.8400,0.7021,0.7857,0.7416,0.5976,0.5999
4,0.8065,0.8617,0.7083,0.7727,0.7391,0.5857,0.5871
Mean,0.8218,0.8737,0.7529,0.7783,0.7642,0.6211,0.6226
Std,0.0324,0.0242,0.0512,0.0521,0.0418,0.0678,0.0682


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8720,0.9030,0.8125,0.8478,0.8298,0.7273,0.7277
1,0.8880,0.9029,0.7708,0.9250,0.8409,0.7556,0.7631
2,0.8320,0.8895,0.7292,0.8140,0.7692,0.6378,0.6402
3,0.8387,0.8179,0.7021,0.8462,0.7674,0.6456,0.6522
4,0.8387,0.8258,0.7083,0.8500,0.7727,0.6493,0.6558
Mean,0.8539,0.8678,0.7446,0.8566,0.7960,0.6831,0.6878
Std,0.0221,0.0379,0.0416,0.0367,0.0323,0.0486,0.0486


      Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC
Fold                                                          
0       0.8720  0.9030  0.8125  0.8478  0.8298  0.7273  0.7277
1       0.8880  0.9029  0.7708  0.9250  0.8409  0.7556  0.7631
2       0.8320  0.8895  0.7292  0.8140  0.7692  0.6378  0.6402
3       0.8387  0.8179  0.7021  0.8462  0.7674  0.6456  0.6522
4       0.8387  0.8258  0.7083  0.8500  0.7727  0.6493  0.6558
Mean    0.8539  0.8678  0.7446  0.8566  0.7960  0.6831  0.6878
Std     0.0221  0.0379  0.0416  0.0367  0.0323  0.0486  0.0486


**Train a single model**

In [ ]:


from pycaret.classification import create_model, tune_model, finalize_model, predict_model

# Example: LightGBM
lgbm = create_model('lightgbm')
tuned_lgbm = tune_model(create_model('lightgbm'), optimize='Accuracy', n_iter=50)
final_lgbm = finalize_model(tuned_lgbm)

# Evaluate on validation (or test) set
predictions = predict_model(final_lgbm, data=X_test)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8480,0.8854,0.8125,0.7959,0.8041,0.6800,0.6801
1,0.8640,0.8807,0.7917,0.8444,0.8172,0.7091,0.7100
2,0.8320,0.9125,0.7292,0.8140,0.7692,0.6378,0.6402
3,0.8306,0.8008,0.7234,0.8095,0.7640,0.6326,0.6351
4,0.8306,0.8192,0.6875,0.8462,0.7586,0.6303,0.6384
Mean,0.8411,0.8597,0.7488,0.8220,0.7826,0.6580,0.6607
Std,0.0132,0.0424,0.0462,0.0199,0.0235,0.0313,0.0296


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8480,0.8854,0.8125,0.7959,0.8041,0.6800,0.6801
1,0.8640,0.8807,0.7917,0.8444,0.8172,0.7091,0.7100
2,0.8320,0.9125,0.7292,0.8140,0.7692,0.6378,0.6402
3,0.8306,0.8008,0.7234,0.8095,0.7640,0.6326,0.6351
4,0.8306,0.8192,0.6875,0.8462,0.7586,0.6303,0.6384
Mean,0.8411,0.8597,0.7488,0.8220,0.7826,0.6580,0.6607
Std,0.0132,0.0424,0.0462,0.0199,0.0235,0.0313,0.0296


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8640,0.9125,0.8333,0.8163,0.8247,0.7137,0.7138
1,0.8640,0.9079,0.7500,0.8780,0.8090,0.7044,0.7097
2,0.8320,0.8892,0.7083,0.8293,0.7640,0.6349,0.6396
3,0.8306,0.8359,0.7234,0.8095,0.7640,0.6326,0.6351
4,0.8468,0.8126,0.7083,0.8718,0.7816,0.6655,0.6740
Mean,0.8475,0.8716,0.7447,0.8410,0.7887,0.6702,0.6744
Std,0.0146,0.0401,0.0469,0.0285,0.0244,0.0339,0.0333


Fitting 5 folds for each of 50 candidates, totalling 250 fits


[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_freq is set=7, subsample_freq=0 will be ignored. Current value: bagging_freq=7
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_freq is set=7, subsample_freq=0 will be ignored. Current value: bagging_freq=7
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9


In [40]:
bagged_lgb = ensemble_model(tuned_lgbm, method='Bagging')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7600,0.7684,0.6458,0.7045,0.6739,0.4846,0.4858
1,0.8400,0.7812,0.7292,0.8333,0.7778,0.6536,0.6572
2,0.7200,0.7784,0.5000,0.6857,0.5783,0.3763,0.3869
3,0.7823,0.7702,0.6383,0.7500,0.6897,0.5236,0.5277
4,0.7742,0.8137,0.5417,0.8125,0.6500,0.4930,0.5151
Mean,0.7753,0.7824,0.6110,0.7572,0.6739,0.5062,0.5145
Std,0.0388,0.0164,0.0813,0.0579,0.0644,0.0889,0.0868




The best model is Catboost. I have been optimizing Catboost models in various ways, and I have also evaluated the performance of models with the top four models as ensemble.


In [30]:
import optuna



The following model is the best model I obtained while tuning myself using the annotation code below. I hope you can refer to it. If you derive results after learning in this model, you can get a score of about 0.8097.


In [31]:
catboost_best = create_model('catboost', nan_mode= 'Min',
                         eval_metric='Logloss',
                         iterations=1000,
                         sampling_frequency='PerTree',
                         leaf_estimation_method='Newton',
                         grow_policy='SymmetricTree',
                         penalties_coefficient=1,
                         boosting_type='Plain',
                         model_shrink_mode='Constant',
                         feature_border_type='GreedyLogSum',                        
                         l2_leaf_reg=3,
                         random_strength=1, 
                         rsm=1, 
                         boost_from_average=False,
                         model_size_reg=0.5, 
                         subsample=0.800000011920929, 
                         use_best_model=False, 
                         class_names=[0, 1],
                         depth=6, 
                         posterior_sampling=False, 
                         border_count=254, 
                         classes_count=0, 
                         auto_class_weights='None',
                         sparse_features_conflict_fraction=0, 
                         leaf_estimation_backtracking='AnyImprovement',
                         best_model_min_trees=1, 
                         model_shrink_rate=0, 
                         min_data_in_leaf=1, 
                         loss_function='Logloss',
                         learning_rate=0.02582800015807152,
                         score_function='Cosine',
                         task_type='CPU',
                         leaf_estimation_iterations=10, 
                         bootstrap_type='MVS',
                         max_leaves=64)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6800,0.7549,0.8125,0.5571,0.6610,0.3773,0.4016
1,0.8240,0.7795,0.7292,0.7955,0.7609,0.6220,0.6235
2,0.7280,0.7332,0.8333,0.6061,0.7018,0.4630,0.4829
3,0.7581,0.7578,0.6383,0.6977,0.6667,0.4774,0.4785
4,0.8065,0.7797,0.6667,0.8000,0.7273,0.5792,0.5850
Mean,0.7593,0.7610,0.7360,0.6913,0.7035,0.5038,0.5143
Std,0.0523,0.0174,0.0771,0.0979,0.0375,0.0872,0.0799




The following code is the process of optimizing the model to a variety of algorithms. I compared them all because they had different access methods and different results. I've locked it up as a chairman now.


In [32]:
catboost = tune_model(create_model('catboost'), choose_better = True, n_iter = 20)


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6800,0.7059,0.4167,0.6250,0.5000,0.2783,0.2907
1,0.8240,0.7731,0.7292,0.7955,0.7609,0.6220,0.6235
2,0.7040,0.7424,0.7500,0.5902,0.6606,0.4047,0.4138
3,0.7742,0.7635,0.6383,0.7317,0.6818,0.5081,0.5109
4,0.8065,0.8008,0.6250,0.8333,0.7143,0.5724,0.5860
Mean,0.7577,0.7572,0.6318,0.7151,0.6635,0.4771,0.4850
Std,0.0565,0.0318,0.1182,0.0943,0.0885,0.1231,0.1207


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7920,0.7726,0.7708,0.7115,0.7400,0.5671,0.5684
1,0.8400,0.7560,0.7292,0.8333,0.7778,0.6536,0.6572
2,0.7840,0.7854,0.7500,0.7059,0.7273,0.5487,0.5494
3,0.7823,0.7784,0.6383,0.7500,0.6897,0.5236,0.5277
4,0.7661,0.7496,0.7083,0.6939,0.7010,0.5090,0.5091
Mean,0.7929,0.7684,0.7193,0.7389,0.7271,0.5604,0.5624
Std,0.0250,0.0135,0.0456,0.0508,0.0310,0.0507,0.0515


Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [33]:
catboost2 = tune_model(create_model('lightgbm'), optimize='Accuracy', 
                       search_library='scikit-optimize', search_algorithm='bayesian', 
                       choose_better = True, n_iter = 20)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8080,0.7812,0.8125,0.7222,0.7647,0.6035,0.6065
1,0.8320,0.7776,0.7292,0.8140,0.7692,0.6378,0.6402
2,0.7360,0.7296,0.7708,0.6271,0.6916,0.4650,0.4726
3,0.7581,0.7624,0.6383,0.6977,0.6667,0.4774,0.4785
4,0.7500,0.7741,0.7083,0.6667,0.6869,0.4791,0.4797
Mean,0.7768,0.7650,0.7318,0.7055,0.7158,0.5326,0.5355
Std,0.0368,0.0188,0.0589,0.0629,0.0426,0.0729,0.0725


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7920,0.7884,0.7708,0.7115,0.7400,0.5671,0.5684
1,0.8400,0.8190,0.7292,0.8333,0.7778,0.6536,0.6572
2,0.7840,0.7873,0.7500,0.7059,0.7273,0.5487,0.5494
3,0.7903,0.8262,0.6383,0.7692,0.6977,0.5393,0.5448
4,0.7742,0.8459,0.5417,0.8125,0.6500,0.4930,0.5151
Mean,0.7961,0.8134,0.6860,0.7665,0.7185,0.5604,0.5670
Std,0.0228,0.0226,0.0852,0.0515,0.0428,0.0527,0.0482


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi

In [26]:
catboost3 = tune_model(create_model('catboost'), optimize='Accuracy',
                       search_library='optuna', search_algorithm='random',
                       choose_better = True, n_iter = 20)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8224,0.9034,0.8260,0.8224,0.8242,0.6448,0.6448
1,0.7836,0.8860,0.7817,0.7874,0.7845,0.5672,0.5672
2,0.8260,0.9108,0.8271,0.8271,0.8271,0.6520,0.6520
3,0.8037,0.9038,0.8186,0.7969,0.8076,0.6074,0.6076
4,0.8129,0.9044,0.8429,0.7973,0.8194,0.6257,0.6268
Mean,0.8098,0.9017,0.8193,0.8062,0.8126,0.6194,0.6197
Std,0.0152,0.0083,0.0204,0.0156,0.0155,0.0304,0.0304


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8246,0.9022,0.8203,0.8297,0.8250,0.6492,0.6492
1,0.7886,0.8859,0.7932,0.7887,0.7909,0.5772,0.5772
2,0.8217,0.9100,0.8186,0.8256,0.8221,0.6434,0.6435
3,0.8030,0.9018,0.8171,0.7967,0.8068,0.6060,0.6062
4,0.8129,0.9013,0.8329,0.8030,0.8177,0.6258,0.6262
Mean,0.8102,0.9003,0.8164,0.8087,0.8125,0.6203,0.6205
Std,0.0131,0.0078,0.0129,0.0162,0.0124,0.0263,0.0263


In [34]:
catboost6 = tune_model(create_model('catboost'), optimize='Accuracy',
                       search_library='optuna', search_algorithm='tpe',
                       choose_better = True, n_iter = 20)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6800,0.7059,0.4167,0.6250,0.5000,0.2783,0.2907
1,0.8240,0.7731,0.7292,0.7955,0.7609,0.6220,0.6235
2,0.7040,0.7424,0.7500,0.5902,0.6606,0.4047,0.4138
3,0.7742,0.7635,0.6383,0.7317,0.6818,0.5081,0.5109
4,0.8065,0.8008,0.6250,0.8333,0.7143,0.5724,0.5860
Mean,0.7577,0.7572,0.6318,0.7151,0.6635,0.4771,0.4850
Std,0.0565,0.0318,0.1182,0.0943,0.0885,0.1231,0.1207


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7040,0.7807,0.3958,0.7037,0.5067,0.3181,0.3450
1,0.8400,0.7773,0.7292,0.8333,0.7778,0.6536,0.6572
2,0.7840,0.7892,0.6875,0.7333,0.7097,0.5380,0.5387
3,0.7823,0.7604,0.5745,0.7941,0.6667,0.5111,0.5259
4,0.7903,0.8124,0.5833,0.8235,0.6829,0.5330,0.5507
Mean,0.7801,0.7840,0.5941,0.7776,0.6687,0.5108,0.5235
Std,0.0436,0.0170,0.1156,0.0508,0.0895,0.1084,0.1007


In [37]:
ensemble = ensemble_model(created[0], method='Bagging')
predict_model(ensemble);

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7040,0.7727,0.3958,0.7037,0.5067,0.3181,0.3450
1,0.8400,0.7844,0.7292,0.8333,0.7778,0.6536,0.6572
2,0.7040,0.7863,0.4375,0.6774,0.5316,0.3296,0.3465
3,0.7500,0.7463,0.4255,0.8333,0.5634,0.4130,0.4588
4,0.7823,0.8176,0.5625,0.8182,0.6667,0.5131,0.5330
Mean,0.7561,0.7815,0.5101,0.7732,0.6092,0.4455,0.4681
Std,0.0514,0.0230,0.1235,0.0682,0.1003,0.1254,0.1183


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,CatBoost Classifier,0.7388,0.8121,0.4466,0.7797,0.5679,0.3999,0.4318


# Interpreting Model
Pycaret provides SHAP. This explains why the model derived the results in this way. The following figure visualizes the shape value of each variable based on the size of the value.

In [ ]:
interpret_model(catboost_best)

# NN

In [ ]:
%%time
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau, LearningRateScheduler, EarlyStopping
from tensorflow.keras.layers import Dense, Input, InputLayer, Add, BatchNormalization, Dropout, Concatenate

from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
import random

from sklearn.model_selection import KFold, StratifiedKFold 
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
import datetime
import math

2026-02-05 23:34:37.745542: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-05 23:34:37.901030: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-05 23:34:39.082549: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


CPU times: user 1.21 s, sys: 196 ms, total: 1.41 s
Wall time: 1.97 s


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, BatchNormalization
import tensorflow as tf

def nn_model_one(input_dim):
    """
    Fully connected neural network for binary classification.

    Parameters
    ----------
    input_dim : int
        Number of features in the input data.

    Returns
    -------
    model : tf.keras.Model
        Compiled Keras model.
    """
    
    activation_func = 'swish'
    inputs = Input(shape=(input_dim,))
    
    x = Dense(1024, kernel_regularizer=tf.keras.regularizers.l2(30e-6),
              activation=activation_func)(inputs)
    x = BatchNormalization()(x)
    
    x = Dense(256, kernel_regularizer=tf.keras.regularizers.l2(30e-6),
              activation=activation_func)(x)
    x = BatchNormalization()(x)
    
    x = Dense(128, kernel_regularizer=tf.keras.regularizers.l2(30e-6),
              activation=activation_func)(x)
    x = BatchNormalization()(x)
    
    x = Dense(1, activation='sigmoid')(x)  # binary output
    
    model = Model(inputs, x)
    return model


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
import tensorflow as tf

def nn_model_two(input_dim):
    """
    Fully connected neural network with Dropout for binary classification.

    Parameters
    ----------
    input_dim : int
        Number of features in the input data.

    Returns
    -------
    model : tf.keras.Model
        Keras model ready for compilation.
    """
    
    dropout_value = 0.025
    activation_func = 'swish'
    
    inputs = Input(shape=(input_dim,))
    
    x = Dense(1024, kernel_regularizer=tf.keras.regularizers.l2(30e-6),
              activation=activation_func)(inputs)
    x = BatchNormalization()(x)
    x = Dropout(dropout_value)(x)
    
    x = Dense(256, kernel_regularizer=tf.keras.regularizers.l2(30e-6),
              activation=activation_func)(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_value)(x)
    
    x = Dense(128, kernel_regularizer=tf.keras.regularizers.l2(30e-6),
              activation=activation_func)(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_value)(x)
    
    x = Dense(8, kernel_regularizer=tf.keras.regularizers.l2(30e-6),
              activation=activation_func)(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_value)(x)
    
    x = Dense(1, activation='sigmoid')(x)  # binary output
    
    model = Model(inputs, x)
    return model


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout, Concatenate
import tensorflow as tf

def nn_model_three(input_dim):
    """
    Neural network with skip connection and Dropout for binary classification.

    Parameters
    ----------
    input_dim : int
        Number of features in the input data.

    Returns
    -------
    model : tf.keras.Model
        Keras model ready for compilation.
    """
    
    L2 = 65e-6
    dropout_value = 0.025
    activation_func = 'swish'
    
    inputs = Input(shape=(input_dim,))
    
    # First dense block
    x0 = Dense(1024, kernel_regularizer=tf.keras.regularizers.l2(L2),
               activation=activation_func)(inputs)
    x0 = BatchNormalization()(x0)
    x0 = Dropout(dropout_value)(x0)
    
    # Second dense block
    x1 = Dense(1024, kernel_regularizer=tf.keras.regularizers.l2(L2),
               activation=activation_func)(x0)
    x1 = BatchNormalization()(x1)
    x1 = Dropout(dropout_value)(x1)
    
    # Third dense block with skip connection
    x1_dense = Dense(64, kernel_regularizer=tf.keras.regularizers.l2(L2),
                     activation=activation_func)(x1)
    x1 = Concatenate()([x1_dense, x0])  # skip connection
    x1 = BatchNormalization()(x1)
    x1 = Dropout(dropout_value)(x1)
    
    # Fourth dense block
    x1 = Dense(16, kernel_regularizer=tf.keras.regularizers.l2(L2),
               activation=activation_func)(x1)
    x1 = BatchNormalization()(x1)
    x1 = Dropout(dropout_value)(x1)
    
    # Output layer
    outputs = Dense(1, activation='sigmoid')(x1)
    
    model = Model(inputs, outputs)
    return model


In [ ]:
architecture = nn_model_one()
architecture.summary()

TypeError: nn_model_one() missing 1 required positional argument: 'input_dim'

In [ ]:
# Model training hyperparameters
BATCH_SIZE         = 128
EPOCHS             = 250 
EPOCHS_COSINEDECAY = 250
DIAGRAMS           = True       # Whether to plot training history
USE_PLATEAU        = True       # Whether to use ReduceLROnPlateau
INFERENCE          = False      # Whether this is just inference mode
VERBOSE            = 0          # Model.fit verbosity
TARGET             = 'Transported'


In [ ]:
def fit_model(X_train, y_train, X_val, y_val, fold=0, model_fn=nn_model_one):
    """
    Trains a neural network on X_train/y_train, evaluates on X_val/y_val.
    Stores history, score, and predictions globally.

    Parameters
    ----------
    X_train : pd.DataFrame or np.array
        Training features
    y_train : pd.Series or np.array
        Training target
    X_val : pd.DataFrame or np.array
        Validation features
    y_val : pd.Series or np.array
        Validation target
    fold : int
        Fold number for K-Fold CV
    model_fn : function
        Function that returns a compiled Keras model given input_dim
    """
    import datetime, math
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.metrics import accuracy_score
    import tensorflow as tf
    from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, LearningRateScheduler, TerminateOnNaN

    start_time = datetime.datetime.now()

    # 1️⃣ Scale features
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)

    # 2️⃣ Callbacks
    callbacks = []
    if USE_PLATEAU:
        lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=8, verbose=VERBOSE)
        es = EarlyStopping(monitor='val_loss', patience=16, verbose=1, mode='min', restore_best_weights=True)
        tm = TerminateOnNaN()
        callbacks = [lr, es, tm]
        epochs = EPOCHS
    else:
        # Cosine decay
        epochs = EPOCHS_COSINEDECAY
        lr_start, lr_end = 0.2, 0.0002
        def cosine_decay(epoch):
            w = (1 + math.cos(epoch / (epochs - 1) * math.pi)) / 2 if epochs > 1 else 1
            return w * lr_start + (1 - w) * lr_end
        lr = LearningRateScheduler(cosine_decay, verbose=0)
        tm = TerminateOnNaN()
        callbacks = [lr, tm]

    # 3️⃣ Build and compile model
    model = model_fn(X_train.shape[1])  # pass number of features
    optimizer_func = tf.keras.optimizers.Adam(learning_rate=0.2)
    model.compile(optimizer=optimizer_func, loss='binary_crossentropy', metrics=['AUC'])

    # 4️⃣ Fit model
    history = model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=epochs,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=VERBOSE,
        callbacks=callbacks
    )

    # 5️⃣ Store history
    history_list.append(history.history)

    # 6️⃣ Evaluate
    y_val_pred = model.predict(X_val_scaled).ravel()
    y_val_pred_binary = [1 if x > 0.5 else 0 for x in y_val_pred]
    score = accuracy_score(y_val, y_val_pred_binary)
    print(f'Fold {fold} | {str(datetime.datetime.now() - start_time)[-12:-7]} | ACC: {score:.5f}')
    score_list.append(score)

    # 7️⃣ Optionally, store test predictions here
    # tst_pred = model.predict(scaler.transform(X_test))  
    # predictions.append(tst_pred)

    return model


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
import numpy as np

# 1️⃣ Initialize storage
history_list = []
score_list   = []
predictions  = []

# 2️⃣ Identify categorical vs numeric columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols     = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 3️⃣ Stratified K-Fold
kf = StratifiedKFold(n_splits=5, random_state=15, shuffle=True)

for fold, (trn_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    print(f"\nStarting fold {fold + 1}")
    
    # Split fold
    X_train_fold, X_val_fold = X_train.iloc[trn_idx], X_train.iloc[val_idx]
    y_train_fold, y_val_fold = y_train.iloc[trn_idx], y_train.iloc[val_idx]
    
    # 4️⃣ Preprocessing: encode categorical + scale numeric
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', MinMaxScaler(), numeric_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
        ]
    )
    
    X_train_scaled = preprocessor.fit_transform(X_train_fold)
    X_val_scaled   = preprocessor.transform(X_val_fold)
    
    # 5️⃣ Call your model training function
    # Note: pass number of features
    model = fit_model(X_train_scaled, y_train_fold, X_val_scaled, y_val_fold, 
                      fold=fold, model_fn=nn_model_one)  # or nn_model_two / nn_model_three

# 6️⃣ Print overall score
print(f'OOF ACC: {np.mean(score_list):.5f}')



Starting fold 1


2026-02-05 23:39:06.657552: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fd1bc0102c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-05 23:39:06.657581: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-02-05 23:39:06.707394: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-05 23:39:06.995661: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2026-02-05 23:39:07.038932: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-05 23:39:08.

Epoch 54: early stopping
Restoring model weights from the end of the best epoch: 38.
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step 
Fold 0 | 00:19 | ACC: 0.76779

Starting fold 2
Epoch 84: early stopping
Restoring model weights from the end of the best epoch: 68.
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step 
Fold 1 | 00:16 | ACC: 0.78720

Starting fold 3
Epoch 62: early stopping
Restoring model weights from the end of the best epoch: 46.
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
Fold 2 | 00:17 | ACC: 0.78577

Starting fold 4
Epoch 65: early stopping
Restoring model weights from the end of the best epoch: 49.
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step 
Fold 3 | 00:19 | ACC: 0.80518

Starting fold 5


2026-02-05 23:40:22.775971: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-05 23:40:22.776122: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-05 23:40:22.776229: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-05 23:40:23.434317: I external/l

Epoch 58: early stopping
Restoring model weights from the end of the best epoch: 42.
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step 
Fold 4 | 00:18 | ACC: 0.80432
OOF ACC: 0.79005


Generating Predictions

In [ ]:
# Populated the prediction on the submission dataset and creates an output file
sub['Transported'] = np.array(predictions).mean(axis = 0)
sub['Transported'] = np.where(sub['Transported'] > 0.5, True, False)
sub.to_csv('my_submission_051922.csv', index = False)

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,...,Cabin_deck,Cabin_side,Cabin_region1,Cabin_region2,Cabin_region3,Cabin_region4,Cabin_region5,Cabin_region6,Cabin_region7,Family_size
0,Earth,False,TRAPPIST-1e,0.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,...,G,S,0,0,1,0,0,0,0,9
1,Earth,True,TRAPPIST-1e,17.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,...,G,S,1,0,0,0,0,0,0,92
2,Earth,True,PSO J318.5-22,35.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,...,G,S,0,0,0,0,1,0,0,1
3,Europa,True,55 Cancri e,26.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,...,D,S,1,0,0,0,0,0,0,1
4,Earth,False,TRAPPIST-1e,13.0,False,0.000000,0.000000,4.110874,0.693147,8.546364,...,G,P,0,0,1,0,0,0,0,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6949,Europa,False,TRAPPIST-1e,43.0,False,0.000000,7.574558,0.000000,0.000000,7.409742,...,B,S,1,0,0,0,0,0,0,1
6950,Earth,False,TRAPPIST-1e,38.0,False,5.214936,5.318120,0.000000,4.709530,5.926926,...,F,P,0,0,0,1,0,0,0,2
6951,Earth,False,PSO J318.5-22,45.0,False,0.693147,2.079442,4.043051,6.419995,0.000000,...,G,S,0,0,0,0,1,0,0,3
6952,Earth,True,PSO J318.5-22,24.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,...,G,S,1,0,0,0,0,0,0,2
